# Kompresja bitmapy RGB

## Zadanie

- Proszę wybrać kolorową bitmapę
- Proszę zamienić bitmapę na 3 macierze Red, Green, Blue (wartości z przedziału 0-255)
- Proszę napisać rekurencyjną kompresje macierzy z wykorzystaniem częściowego SVD dla wybranych parametrów $\sigma$= najmniejsza wartość osobliwa (wyrzucamy mniejsze) i $b$= maksymalny rank (liczba wartości osobliwych)
- Proszę zaimplementować rysowawcz skompresowanej macierzy
- Proszę zaimplementować rysowacz skompresowanej bitmapy

## Pseudokod

```text
CreateTree(mat, r, eps):
    m, n = mat.shape
    U, D, V = truncatedSVD(mat, r + 1)
    if D(r + 1, r + 1) < eps:
        v = CompressMatrix(mat, U, D, V, r)
    else
        create new node v
        Append(v, CreateTree(mat[:m//2, :n//2], r, eps))
        Append(v, CreateTree(mat[:m//2, n//2:], r, eps))
        Append(v, CreateTree(mat[m//2:, :n//2], r, eps))
        Append(v, CreateTree(mat[m//2:, n//2:], r, eps))
    return v
```

```text
CompressMatrix(mat, U, D, V, r):
    if all values in D are zeros:
        create new node v
        v.rank = 0
        v.size = size(mat)
        return v
    sigma = diag(D)
    rank = r
    create new node v
    v.rank = rank
    v.singular_values = sigma[:rank]
    v.U = U[:, :rank]
    v.V = D[:rank, :rank] @ V[:rank, :]
    v.children = []
    v.size = size(mat)
    return v
```


In [ ]:
from dataclasses import dataclass
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.image import imread
from scipy.sparse.linalg import svds

np.random.seed(0)

In [ ]:
import numpy.typing as npt
from typing import NewType, Any, Optional

ImageArray = NewType("ImageArray", npt.NDArray[np.float32])
ChannelArray = npt.NDArray[np.float32]
UArray = npt.NDArray[np.float32]
SArray = npt.NDArray[np.float32]
VArray = npt.NDArray[np.float32]

In [ ]:
def load_image(file_path: str) -> ImageArray:
    image = imread(file_path).astype(np.float32) / 255
    assert image.ndim == 3 and image.shape[2] == 3, "Image must have shape (H, W, 3)"
    return ImageArray(image)

In [ ]:
def extract_rgb(
    image: ImageArray,
) -> tuple[ChannelArray, ChannelArray, ChannelArray]:
    return image[..., 0], image[..., 1], image[..., 2]

In [ ]:
def plot_channel(ax: Any, channel: ChannelArray, title: str) -> None:
    ax.imshow(channel, cmap="gray")
    ax.set_title(title)

In [ ]:
def plot_image(image: ImageArray, title: str) -> None:
    _, axs = plt.subplots(2, 2, figsize=(10, 8), layout="tight")
    r, g, b = extract_rgb(image)
    plot_channel(axs[0, 0], image, "Original image")
    plot_channel(axs[0, 1], r, "Red channel")
    plot_channel(axs[1, 0], g, "Green channel")
    plot_channel(axs[1, 1], b, "Blue channel")

In [ ]:
@dataclass
class CompressedNode:
    rank: int
    u: npt.NDArray[np.float32]
    s: npt.NDArray[np.float32]
    v: npt.NDArray[np.float32]
    top_left: Optional["CompressedNode"] = None
    top_right: Optional["CompressedNode"] = None
    bottom_left: Optional["CompressedNode"] = None
    bottom_right: Optional["CompressedNode"] = None

In [ ]:
@dataclass
class CompressedTree:
    root: CompressedNode

    def __str__(self) -> str:
        return f"CompressedTree({self.root})"

In [ ]:
def partial_svd(
    mat: ChannelArray, k: int
) -> tuple[UArray, SArray, VArray]:
    return svds(mat, k=k, solver="propack")  # type: ignore

In [ ]:
def create_tree(mat: ChannelArray, r: int, eps: int) -> CompressedTree:
    u, s, v = partial_svd(mat, r)
    print(type(u[0][0]), type(s[0]), type(v[0][0]))
    node = CompressedNode(r, u, s, v)

    return CompressedTree(node)

In [ ]:
image = load_image(r"data/100zloty.jpg")
# plot_image(image, "Original image")

In [ ]:
r, g, b = extract_rgb(image)
create_tree(r, 10, 0.1)